# Profiles (Sinopia templates)
Profiles are the resource templates Sinopia and Marva build their cataloging forms from. Find one, see which properties it defines, and (locally) create, edit and delete a copy. Runs against the notebook's environment: your local stack, or a deployed one (dev, stage), which it only reads. Pick it on the Notebooks page, or call `target("dev")`.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

## Find profiles
`/search/profile` searches their text; `nested=exclude` leaves out templates that are only used inside others.

In [ ]:
import pandas as pd

SV, RDFS = "http://sinopia.io/vocabulary/", "http://www.w3.org/2000/01/rdf-schema#"

def first(node, key):
    values = node.get(key) or [{}]
    return values[0].get("@value") or values[0].get("@id")

def template_of(profile):
    """The profile's ResourceTemplate node"""
    return next((n for n in profile["data"] if SV + "ResourceTemplate" in n.get("@type", [])), {})

QUERY = "monograph"
found = bc.get("/api/search/profile", params={"q": QUERY, "limit": 15}).json()
print(found["total"], "profiles match", repr(QUERY))
pd.DataFrame([
    {"label": first(template_of(p), RDFS + "label"), "id": first(template_of(p), SV + "hasResourceId"),
     "class": first(template_of(p), SV + "hasClass"), "uuid": p["uuid"]}
    for p in found["results"]
])

## One profile's properties
The first result, or set `PROFILE_UUID`.

In [ ]:
listed = found["results"] or bc.get("/api/profiles/", params={"limit": 1}).json()["profiles"]
PROFILE_UUID = listed[0]["uuid"] if listed else None
profile = bc.get(f"/api/profiles/{PROFILE_UUID}").json() if PROFILE_UUID else None
properties = [n for n in profile["data"] if SV + "PropertyTemplate" in n.get("@type", [])] if profile else []
print(first(template_of(profile), RDFS + "label"), "-", len(properties), "properties") if profile else print("No profiles in this environment yet: the last section creates one.")
pd.DataFrame([
    {"label": first(p, RDFS + "label"), "property": first(p, SV + "hasPropertyUri"),
     "type": (first(p, SV + "hasPropertyType") or "").rsplit("/", 1)[-1],
     "repeatable": any(a.get("@id", "").endswith("repeatable") for a in p.get(SV + "hasPropertyAttribute", [])),
     "required": any(a.get("@id", "").endswith("required") for a in p.get(SV + "hasPropertyAttribute", []))}
    for p in properties
])

In [ ]:
pd.DataFrame(bc.get(f"/api/profiles/{PROFILE_UUID}/versions").json()["versions"]) if PROFILE_UUID else "No profile."

## Create, edit and delete a profile
Local stack only: copies the profile above (or a minimal template, if there's none) under a new id, renames it, and deletes it.

In [ ]:
import json, uuid

bc.require_writable("Creating a profile")
AUTH = bc.auth_headers()
# A copy of the profile above, or a minimal template when there's none
source = profile["data"] if profile else [{"@id": "", "@type": [SV + "ResourceTemplate"], SV + "hasClass": [{"@id": "http://id.loc.gov/ontologies/bibframe/Work"}]}]
copy = json.loads(json.dumps(source))
tpl = next(n for n in copy if SV + "ResourceTemplate" in n.get("@type", []))
tpl[SV + "hasResourceId"] = [{"@value": f"notebook:copy:{uuid.uuid4().hex[:8]}"}]
tpl[RDFS + "label"] = [{"@value": "Notebook copy of " + (first(tpl, RDFS + "label") or "a profile")}]

r = bc.post("/api/profiles/", json={"data": json.dumps(copy)}, headers=AUTH)
show(r, 200)
new = r.json()
print("Minted:", new["uri"])

tpl = next(n for n in new["data"] if SV + "ResourceTemplate" in n.get("@type", []))
tpl[RDFS + "label"] = [{"@value": "Notebook copy (renamed)"}]
show(bc.put(f"/api/profiles/{new['uuid']}", json={"data": json.dumps(new["data"])}, headers=AUTH), 200)
print(len(bc.get(f"/api/profiles/{new['uuid']}/versions").json()["versions"]), "versions")
print("delete:", bc.delete(f"/api/profiles/{new['uuid']}", headers=AUTH).status_code)